# N108 · 划分DP与最后一段

**目标：** 用最后一段边界枚举前缀最优并计算段成本。

**先修：** N098, N031, N107。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** dp[i]/dp[k][i]；段成本；前缀优化；最少/恰好K段；无效状态。

## 从问题到算法

划分DP枚举最后一段起点。先规定段数是否固定、段长是否有上限，再决定状态维度。最大分块和是前缀最优；回文划分先预计算每段改成回文所需替换次数；工作日安排必须每天非空，不能用零成本空段凑天数。

## 最小实现

**本章接口：** `max_sum_after_partitioning(arr, k)`、`palindrome_partition_k(s, k)`、`min_difficulty(jobs, d)`

In [1]:
def max_sum_after_partitioning(arr,k):
    if k<=0: raise ValueError('positive segment limit required')
    dp=[0]+[float('-inf')]*len(arr)
    for i in range(1,len(arr)+1):
        maximum=float('-inf')
        for length in range(1,min(k,i)+1):
            maximum=max(maximum,arr[i-length]); dp[i]=max(dp[i],dp[i-length]+maximum*length)
    return dp[-1]

def palindrome_partition_k(s,k):
    n=len(s)
    if k==0: return 0 if not n else -1
    if k<0 or k>n: return -1
    cost=[[0]*n for _ in range(n)]
    for gap in range(1,n):
        for l in range(n-gap):
            r=l+gap; cost[l][r]=(cost[l+1][r-1] if gap>1 else 0)+(s[l]!=s[r])
    prev=[0]+[float('inf')]*n
    for groups in range(1,k+1):
        dp=[float('inf')]*(n+1)
        for i in range(groups,n+1): dp[i]=min(prev[j]+cost[j][i-1] for j in range(groups-1,i))
        prev=dp
    return prev[n]

def min_difficulty(jobs,d):
    n=len(jobs)
    if d==0: return 0 if not n else -1
    if n<d or d<0: return -1
    prev=[0]+[float('inf')]*n
    for day in range(1,d+1):
        dp=[float('inf')]*(n+1)
        for i in range(day,n+1):
            maximum=float('-inf')
            for j in range(i-1,day-2,-1): maximum=max(maximum,jobs[j]); dp[i]=min(dp[i],prev[j]+maximum)
        prev=dp
    return prev[n]

## 正确性、前提与复杂度

每个合法划分都有唯一最后切点j；前缀最优加最后段代价枚举了全部划分。段数g的前缀至少长g，限制j≥g−1保证每段非空。回文修复成本等于相对位置不相等的对数，各对独立。

**代价：** 最大分块和O(nk)时间O(n)空间；回文划分O(n²+kn²)时间O(n²)空间；工作安排O(dn²)时间O(n)辅助。固定段数不合法时返回−1。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,15,7,9,2,5,10]; show_table(['前缀长度','最大分块和，k=3'],[(i,max_sum_after_partitioning(a[:i],3)) for i in range(len(a)+1)])

前缀长度,最大分块和，k=3
0,0
1,1
2,30
3,45
4,54
5,63
6,72
7,84


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert max_sum_after_partitioning([1,15,7,9,2,5,10],3)==84
assert min_difficulty([6,5,4,3,2,1],2)==7 and min_difficulty([1,2],3)==-1
assert palindrome_partition_k('abc',2)==1 and palindrome_partition_k('abc',3)==0
from itertools import combinations,product

def partitions(a,k):
    for cuts in combinations(range(1,len(a)),k-1):
        ends=(0,)+cuts+(len(a),)
        yield [a[ends[i]:ends[i+1]] for i in range(k)]
for a in product(range(3),repeat=5):
    s=''.join(map(str,a))
    for k in range(1,6):
        ps=list(partitions(a,k)); sp=list(partitions(s,k))
        assert min_difficulty(a,k)==min(sum(max(x) for x in p) for p in ps)
        assert palindrome_partition_k(s,k)==min(sum(sum(x[i]!=x[-1-i] for i in range(len(x)//2)) for x in p) for p in sp)
    for limit in (1,2,3):
        want=max(sum(max(x)*len(x) for x in p) for k in range(1,6) for p in partitions(a,k) if all(len(x)<=limit for x in p))
        assert max_sum_after_partitioning(a,limit)==want
print('N108: 所有本章断言通过')

N108: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 返回划分边界。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较恰好d段与至多d段。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1043. Partition Array for Maximum Sum（canonical）
- 1278. Palindrome Partitioning III（canonical）
- 1335. Minimum Difficulty of a Job Schedule（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。